# Compound AI patterns — 18 mini assessments

Three exercises per pattern: **decomposition, refinement, reformulation, routing, parallelization, and tool use**.

These are focused skill drills, not full interview simulations or known Distyl questions. Each takes about **10–15 minutes**; a few advanced exercises may take 20. Work on one exercise at a time. No solutions, automated test suites, or companion files are included.

## How to practice

1. Run the shared setup once. Read one problem and run its supplied data cell.
2. Say aloud what each model call should receive and return.
3. Fill in the TODO function using ordinary Python. Keep the public function signature.
4. Uncomment the example call. Inspect the result and `model.calls` against the manual success criteria.
5. Before rerunning, call `model.reset()` so recorded responses start from the beginning.

Each exercise reuses names such as `model` and `TOOLS`. Run its own data cell immediately before attempting it; after Run All, those names refer to the last exercise. Run All validates starter definitions but does not solve the TODO functions. A computer running Python 3.10 or newer is required.

**Scaffolding is intentional.** The target pattern and model operations are supplied so you can focus on implementing that pattern. Intermediate model responses are fixtures, not Python solutions. Once these drills feel easy, return to the full assessments where you choose the architecture yourself.

## The model interface

```python
text = model(prompt=json.dumps(payload), system_prompt="OP:operation_name")
data = json.loads(text)
```

Each problem documents its available operations and required payload fields. `model` is a stand-in for `call_llm`: it returns **text**, including JSON text. The `OP:` tags are a local mock convention, not a real LLM API feature. In production, the system prompt would describe the task and required output.

Responses are recorded, not intelligent. The mock checks payload keys but cannot judge prompt quality or whether every value is semantically correct. It cannot simulate open-ended inputs or prove that an architecture improves quality. Use the log to inspect what you actually passed between calls. To practice with a live model later, replace the supplied callable while preserving its interface; do API setup outside the timer.

## Contents

| Pattern | Exercises |
|---|---|
| Decomposition | D1 Event preparation · D2 Product comparison · D3 Dependency-aware planning |
| Refinement | R1 Missing facts · R2 Format repair · R3 Bounded revision loop |
| Reformulation | F1 Resolve references · F2 Generate search queries · F3 Preserve ambiguity |
| Routing | G1 Specialist selection · G2 Batch dispatch · G3 Confidence fallback |
| Parallelization | P1 Independent summaries · P2 Independent perspectives · P3 Partial failure |
| Tool use | T1 One tool · T2 Tool-result loop · T3 Reject and recover |

In [ ]:
import copy
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

class RecordedModel:
    """Thread-safe text-response fixture; required payload keys are checked.

    responses: operation -> list of successive return values
    required: operation -> names of mandatory fields in the JSON prompt
    delays: optional operation -> seconds to simulate a slow call
    Put RuntimeError(...) in a response list to simulate a failed call.
    """
    def __init__(self, responses, required=None, delays=None):
        self._initial = responses
        self.required = required or {}
        self.delays = delays or {}
        self._lock = threading.Lock()
        self.reset()

    def reset(self):
        with self._lock:
            self._responses = copy.deepcopy(self._initial)
            self.calls = []

    def __call__(self, prompt, system_prompt=''):
        op = system_prompt.removeprefix('OP:')
        payload = json.loads(prompt)
        if not isinstance(payload, dict):
            raise ValueError('Prompt must encode a JSON object.')
        missing = set(self.required.get(op, [])) - set(payload)
        if missing:
            raise ValueError(f'{op}: missing payload keys {sorted(missing)}')
        with self._lock:
            if op not in self._responses or not self._responses[op]:
                raise RuntimeError(f'No recorded response left for {op}; inspect the operations or reset the model.')
            response = self._responses[op].pop(0)
            record = {'operation': op, 'payload': copy.deepcopy(payload), 'started': time.perf_counter()}
            self.calls.append(record)
        time.sleep(self.delays.get(op, 0))
        record['finished'] = time.perf_counter()
        if isinstance(response, Exception):
            raise response
        return response if isinstance(response, str) else json.dumps(response)

print('Setup ready. No API keys, external services, or extra packages required.')

# 1. Decomposition

Break a larger objective into smaller work units, carry out those units, and combine their results. Decide what information each unit needs and which units depend on earlier results. Unlike routing, you usually perform several parts rather than choosing just one path.

## D1 — Event preparation

**Pattern:** Decomposition · **Time:** 10–15 minutes

A team requests an event preparation note covering venue, equipment, and staffing. Use a model-generated task list, obtain an answer for each task, then produce one combined note. Keep this first drill sequential; concurrency is reserved for P1–P3. Return `{"tasks": [...], "answers": {...}, "final_note": "..."}`.

**Supplied model operations**

- `plan`: payload `{request}` → `{tasks: [{id, question}]}`.
- `answer_venue`, `answer_equipment`, `answer_staffing`: payload `{request, task}` → `{answer}`. Choose the operation from the task ID, not list position.
- `combine`: payload `{request, answers}` → `{final_note}`.

**Manual success criteria**

- Use five calls on this fixture: plan, three task answers, and combination.
- Store answers under task IDs and pass all three answers to `combine`.
- Read the task list from the model response rather than hardcoding the three answers.

In [ ]:
REQUEST = "Prepare a note for a 12-person workshop: venue, equipment, and staffing."
model = RecordedModel({
 'plan': [{'tasks': [{'id':'venue','question':'What venue is needed?'}, {'id':'equipment','question':'What equipment is needed?'}, {'id':'staffing','question':'Who should support the event?'}]}],
 'answer_venue': [{'answer':'A room with 12 seats and accessible entry.'}],
 'answer_equipment': [{'answer':'One projector and a shared whiteboard.'}],
 'answer_staffing': [{'answer':'One facilitator and one check-in helper.'}],
 'combine': [{'final_note':'Reserve a 12-seat accessible room, prepare a projector and whiteboard, and assign a facilitator and check-in helper.'}]
}, required={'plan':['request'], **{f'answer_{x}':['request','task'] for x in ['venue','equipment','staffing']}, 'combine':['request','answers']})

In [ ]:
def prepare_event(request, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement D1")

# Run after implementing:
# model.reset()
# result = prepare_event(REQUEST, model)
# print(result)
# print(model.calls)

**Explain after finishing:** What would break if you assumed the plan always returned tasks in the same order?

## D2 — Compare by criterion

**Pattern:** Decomposition · **Time:** 10–15 minutes

Compare two devices on battery life and portability. The model selects criteria. Analyze each criterion against the supplied facts, then assemble a comparison keyed by criterion. The combination step may be Python; no synthesis call is required. Return a list of `{criterion, comparison, preferred}` rows.

**Supplied model operations**

- `criteria`: payload `{request}` → `{criteria: [string]}`.
- `compare_battery`, `compare_portability`: payload `{criterion, products}` → `{comparison, preferred}`.

**Manual success criteria**

- Obtain criteria from the planning call and process every criterion.
- Supply both product records to each comparison call.
- Preserve the criterion with each result; do not invent an overall winner.

In [ ]:
REQUEST = "Compare Atlas and Birch for commuting, considering battery and portability."
PRODUCTS = [{'name':'Atlas','battery_hours':11,'weight_kg':1.8}, {'name':'Birch','battery_hours':8,'weight_kg':1.2}]
model = RecordedModel({
 'criteria':[{'criteria':['battery','portability']}],
 'compare_battery':[{'comparison':'Atlas lasts longer: 11 versus 8 hours.','preferred':'Atlas'}],
 'compare_portability':[{'comparison':'Birch is lighter: 1.2 versus 1.8 kg.','preferred':'Birch'}]
}, required={'criteria':['request'], 'compare_battery':['criterion','products'], 'compare_portability':['criterion','products']})

In [ ]:
def compare_products(request, products, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement D2")

# Run after implementing:
# model.reset()
# result = compare_products(REQUEST, PRODUCTS, model)
# print(result)
# print(model.calls)

**Explain after finishing:** Why is criterion-by-criterion decomposition useful here, and what overhead does it add?

## D3 — Respect task dependencies

**Pattern:** Decomposition · **Time:** 10–15 minutes

Prepare an onboarding plan where account setup depends on an approved role, and training depends on account setup. The task list is deliberately unordered. Execute only tasks whose dependencies have completed, passing their results as context. Return results keyed by task ID. If a pass makes no progress, raise `ValueError` for an unresolved dependency or cycle.

**Supplied model operations**

- `plan`: payload `{request}` → `{tasks: [{id, depends_on}]}`.
- `do_role`, `do_account`, `do_training`: payload `{request, task, dependency_results}` → `{result}`.

**Manual success criteria**

- Complete role before account and account before training despite the supplied order.
- Pass actual dependency outputs to later calls.
- Explain how your no-progress check handles a cycle; do not run a cyclic loop indefinitely.

In [ ]:
REQUEST = "Plan onboarding for a new analyst."
model = RecordedModel({
 'plan':[{'tasks':[{'id':'training','depends_on':['account']}, {'id':'role','depends_on':[]}, {'id':'account','depends_on':['role']}]}],
 'do_role':[{'result':'Approved role: analyst.'}],
 'do_account':[{'result':'Analyst account prepared.'}],
 'do_training':[{'result':'Account-specific training scheduled.'}]
}, required={'plan':['request'], **{f'do_{x}':['request','task','dependency_results'] for x in ['role','account','training']}})

In [ ]:
def plan_onboarding(request, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement D3")

# Run after implementing:
# model.reset()
# result = plan_onboarding(REQUEST, model)
# print(result)
# print(model.calls)

**Explain after finishing:** Which tasks could run independently if a future plan included several dependency-free tasks?

# 2. Refinement

Generate an initial output, obtain specific feedback, then improve it. Feedback may come from another model call or a deterministic Python check. Define when the process stops.

## R1 — Repair an incomplete notice

**Pattern:** Refinement · **Time:** 10–15 minutes

Draft an office notice that must include a date, time, and location. Ask for feedback, then request one revision using that feedback. Return `{draft, feedback, revised}`. This drill has exactly one review and one revision.

**Supplied model operations**

- `draft`: payload `{facts}` → `{text}`.
- `review`: payload `{facts, draft}` → `{issues: [string]}`.
- `revise`: payload `{facts, draft, issues}` → `{text}`.

**Manual success criteria**

- The review receives the actual initial text and original facts.
- The revision receives both the draft and the review issues.
- Keep the initial draft available so you can compare versions.

In [ ]:
FACTS = {'date':'June 12','time':'14:00','location':'Room Cedar','purpose':'Team workshop'}
model = RecordedModel({
 'draft':[{'text':'Join our team workshop at 14:00 in Room Cedar.'}],
 'review':[{'issues':['The notice omits June 12.']}],
 'revise':[{'text':'Join our team workshop on June 12 at 14:00 in Room Cedar.'}]
}, required={'draft':['facts'], 'review':['facts','draft'], 'revise':['facts','draft','issues']})

In [ ]:
def refine_notice(facts, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement R1")

# Run after implementing:
# model.reset()
# result = refine_notice(FACTS, model)
# print(result)
# print(model.calls)

**Explain after finishing:** How would you tell whether the revision preserved correct information while fixing the omission?

## R2 — Repair using Python feedback

**Pattern:** Refinement · **Time:** 10–15 minutes

Extract a product and quantity. The first response has the wrong type for quantity. Validate that the output is a dictionary with a nonempty string `product` and a nonnegative integer `quantity` (a Boolean does not count). On failure, make one repair call with the raw response and specific error. Return a validated dictionary or raise `ValueError`; do not loop indefinitely.

**Supplied model operations**

- `extract`: payload `{text}` → JSON text.
- `repair`: payload `{text, previous_output, error}` → corrected JSON text.

**Manual success criteria**

- Reject string and Boolean quantities rather than coercing them silently.
- Report the actual failed requirement in the repair payload.
- Allow at most two calls total. Revalidate the repair before returning.

In [ ]:
TEXT = "Please order four desk lamps."
model = RecordedModel({
 'extract':['{"product":"desk lamp","quantity":"four"}'],
 'repair':[{'product':'desk lamp','quantity':4}]
}, required={'extract':['text'], 'repair':['text','previous_output','error']})

In [ ]:
def extract_with_repair(text, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement R2")

# Run after implementing:
# model.reset()
# result = extract_with_repair(TEXT, model)
# print(result)
# print(model.calls)

**Explain after finishing:** What is different about deterministic format feedback versus a model judging factual correctness?

## R3 — Stop a revision loop

**Pattern:** Refinement · **Time:** 10–15 minutes

Produce a handoff note, review it, and revise only while the reviewer finds problems. Allow at most two revisions. Stop immediately when `passed` is true. Return `{text, revisions, passed}`; if still failing at the limit, return `passed=false` with the last text.

**Supplied model operations**

- `draft`: payload `{request}` → `{text}`.
- `review`: payload `{request, text}` → `{passed: boolean, issues: [string]}`.
- `revise`: payload `{request, text, issues}` → `{text}`.

**Manual success criteria**

- This fixture needs four calls and one revision; no second revision should happen.
- Count revisions separately from review calls.
- The cap applies even if a reviewer never approves the output.

In [ ]:
REQUEST = "Create a handoff note with an owner and next action."
model = RecordedModel({
 'draft':[{'text':'Follow up on the migration.'}],
 'review':[{'passed':False,'issues':['Missing owner.']}, {'passed':True,'issues':[]}],
 'revise':[{'text':'Owner: Priya. Next action: follow up on the migration.'}]
}, required={'draft':['request'], 'review':['request','text'], 'revise':['request','text','issues']})

In [ ]:
def bounded_refinement(request, model, max_revisions=2):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement R3")

# Run after implementing:
# model.reset()
# result = bounded_refinement(REQUEST, model)
# print(result)
# print(model.calls)

**Explain after finishing:** How could repeated self-review make an answer worse or waste calls?

# 3. Reformulation

Rewrite an input into a form more useful for a downstream task while preserving its meaning. Rewriting is not the same as answering. Preserve unresolved uncertainty instead of silently choosing an interpretation.

## F1 — Resolve references from context

**Pattern:** Reformulation · **Time:** 10–15 minutes

Rewrite the follow-up question as a standalone question using the conversation history. Then pass that standalone question to an answering call. Return `{standalone_question, answer}`. Preserve the time period and entities from the supplied context.

**Supplied model operations**

- `rewrite`: payload `{history, follow_up}` → `{standalone_question}`.
- `answer`: payload `{question, facts}` → `{answer}`.

**Manual success criteria**

- The second call receives the rewritten question, not the original ambiguous follow-up.
- Supply the facts to the answering call; rewriting alone does not provide evidence.
- Keep the standalone question in the output for inspection.

In [ ]:
HISTORY = [{'role':'user','text':'Compare the Q1 and Q2 support backlogs.'}, {'role':'assistant','text':'The backlog was 120 tickets in Q1 and 90 in Q2.'}]
FOLLOW_UP = "How much did it drop?"
FACTS = {'Q1_backlog':120,'Q2_backlog':90}
model = RecordedModel({
 'rewrite':[{'standalone_question':'How many support tickets did the backlog decrease by from Q1 to Q2?'}],
 'answer':[{'answer':'The support backlog decreased by 30 tickets.'}]
}, required={'rewrite':['history','follow_up'], 'answer':['question','facts']})

In [ ]:
def answer_follow_up(history, follow_up, facts, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement F1")

# Run after implementing:
# model.reset()
# result = answer_follow_up(HISTORY, FOLLOW_UP, FACTS, model)
# print(result)
# print(model.calls)

**Explain after finishing:** What information would be lost if you rewrote the follow-up without its history?

## F2 — Create useful search queries

**Pattern:** Reformulation · **Time:** 10–15 minutes

Turn a vague request into precise search queries, normalize whitespace, remove exact duplicates, and keep at most three. Return `{queries, results}` using the supplied deterministic search tool. Do not write an answer to the request in this drill.

**Supplied model operations**

- `rewrite_queries`: payload `{request}` → `{queries: [string]}`.
- `search(query)` is supplied Python infrastructure, not a model call.

**Manual success criteria**

- Return three distinct nonempty queries in their original order after normalization.
- Call `search` once per retained query and preserve the query/result association.
- Do not silently add a guessed date, system name, or technology.

In [ ]:
REQUEST = "Find notes about our old login setup and why we changed it."
SEARCH_INDEX = {
 'legacy authentication migration rationale':['DOC-11'],
 'previous login architecture':['DOC-12'],
 'authentication migration decision record':['DOC-13']
}
def search(query):
    return SEARCH_INDEX.get(query, [])
model = RecordedModel({
 'rewrite_queries':[{'queries':[' legacy authentication migration rationale ', 'previous login architecture', 'legacy authentication migration rationale', 'authentication migration decision record']}]
}, required={'rewrite_queries':['request']})

In [ ]:
def reformulate_and_search(request, model, search):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement F2")

# Run after implementing:
# model.reset()
# result = reformulate_and_search(REQUEST, model, search)
# print(result)
# print(model.calls)

**Explain after finishing:** Why might several precise queries retrieve better evidence than one vague query?

## F3 — Keep ambiguity visible

**Pattern:** Reformulation · **Time:** 10–15 minutes

Reformulate a request with an ambiguous pronoun. The model returns candidate interpretations and whether clarification is needed. If clarification is needed, return `{status: "needs_clarification", question, interpretations}` and do not call `execute_request`. Otherwise return `{status: "ready", request}` from the sole interpretation. This exercise proposes a clarified request; it sends nothing.

**Supplied model operations**

- `interpret`: payload `{request, context}` → `{needs_clarification, interpretations, clarification_question}`.
- `execute_request` is a supplied recording function; it must not run on the ambiguous fixture.

**Manual success criteria**

- Preserve both possible interpretations and return the blocking question.
- `EXECUTIONS` remains empty for this fixture.
- Explain how your ready branch rejects zero or multiple interpretations rather than selecting arbitrarily.

In [ ]:
REQUEST = "Send it to Jordan after the review."
CONTEXT = {'documents':['budget memo','vendor summary'], 'people':['Jordan Lee','Jordan Patel']}
EXECUTIONS = []
def execute_request(request):
    EXECUTIONS.append(request)
    return request
model = RecordedModel({
 'interpret':[{'needs_clarification':True,'interpretations':['Send the budget memo to Jordan Lee.','Send the vendor summary to Jordan Patel.'], 'clarification_question':'Which document and which Jordan do you mean?'}]
}, required={'interpret':['request','context']})

In [ ]:
def clarify_request(request, context, model, execute_request):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement F3")

# Run after implementing:
# model.reset()
# result = clarify_request(REQUEST, CONTEXT, model, execute_request)
# print(result)
# print(model.calls)

**Explain after finishing:** When does rewriting become inventing user intent?

# 4. Routing

Classify an input and choose the appropriate specialist or processing path. Do not run every specialist. Define a fallback when the route is unknown or uncertain.

## G1 — Choose one specialist

**Pattern:** Routing · **Time:** 10–15 minutes

Classify a support message and send it to exactly one specialist. Allowed routes are `billing`, `technical`, and `general`. Reject an unknown route with `ValueError`. Return `{route, response}`.

**Supplied model operations**

- `classify`: payload `{message}` → `{route}`.
- `billing`, `technical`, `general`: payload `{message}` → `{response}`. Only the selected operation should be called.

**Manual success criteria**

- Make one classification call and one billing call on the sample.
- Pass the original message to the specialist.
- Leave unselected specialist responses unused.

In [ ]:
MESSAGE = "I was charged twice for my subscription."
model = RecordedModel({
 'classify':[{'route':'billing'}],
 'billing':[{'response':'Please provide the two transaction references so Billing can investigate.'}],
 'technical':[{'response':'Technical support response.'}],
 'general':[{'response':'General support response.'}]
}, required={x:['message'] for x in ['classify','billing','technical','general']})

In [ ]:
def route_message(message, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement G1")

# Run after implementing:
# model.reset()
# result = route_message(MESSAGE, model)
# print(result)
# print(model.calls)

**Explain after finishing:** How is this different from decomposing a request and solving all its parts?

## G2 — Route a batch without losing IDs

**Pattern:** Routing · **Time:** 10–15 minutes

Classify two tickets in one model call, then invoke the selected specialist separately for each ticket. Routing rows may arrive out of order. Return results in the original ticket order as `{ticket_id, route, response}` rows. Validate that every ticket has exactly one known route.

**Supplied model operations**

- `classify_batch`: payload `{tickets}` → `{routes: [{ticket_id, route}]}`.
- `billing`, `technical`, `general`: payload `{ticket}` → `{response}`.

**Manual success criteria**

- Match routing rows by ticket ID, not position.
- Return A then B even though classification returns B then A.
- Reject missing, duplicate, or invented ticket IDs.

In [ ]:
TICKETS = [{'ticket_id':'A','message':'I cannot log in.'}, {'ticket_id':'B','message':'My invoice total is wrong.'}]
model = RecordedModel({
 'classify_batch':[{'routes':[{'ticket_id':'B','route':'billing'},{'ticket_id':'A','route':'technical'}]}],
 'technical':[{'response':'Check the sign-in error and account status.'}],
 'billing':[{'response':'Review the invoice line items.'}],
 'general':[{'response':'General support response.'}]
}, required={'classify_batch':['tickets'], **{x:['ticket'] for x in ['billing','technical','general']}})

In [ ]:
def route_batch(tickets, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement G2")

# Run after implementing:
# model.reset()
# result = route_batch(TICKETS, model)
# print(result)
# print(model.calls)

**Explain after finishing:** What can go wrong if you zip model output with the input list?

## G3 — Fall back when routing is uncertain

**Pattern:** Routing · **Time:** 10–15 minutes

Route a message only when the returned route is allowed and confidence is at least 0.8. Otherwise return `{route: "human_review", reason, response: null}` without a specialist call. Do not treat model confidence as a calibrated probability; the threshold is merely this drill’s dispatch rule.

**Supplied model operations**

- `classify`: payload `{message}` → `{route, confidence, reason}`.
- Specialist operations have the same interface as G1; the uncertain fixture should call none.

**Manual success criteria**

- This fixture returns human_review after one call.
- Unknown routes and low confidence both trigger fallback.
- Validate confidence as a numeric value between zero and one; reject malformed types.

In [ ]:
MESSAGE = "The charge failed, but money disappeared and the app froze."
model = RecordedModel({
 'classify':[{'route':'technical','confidence':0.55,'reason':'Both payment and application failure may be involved.'}],
 'technical':[{'response':'A technical-only answer.'}]
}, required={'classify':['message'],'technical':['message']})

In [ ]:
def route_with_fallback(message, model, threshold=0.8):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement G3")

# Run after implementing:
# model.reset()
# result = route_with_fallback(MESSAGE, model)
# print(result)
# print(model.calls)

**Explain after finishing:** How would you choose a useful threshold with real labeled routing examples?

# 5. Parallelization

Execute independent work concurrently, then combine the results. Threads are sufficient for these blocking mock calls. Parallel work should receive the same unchanged input where appropriate. Do not parallelize a task that needs another task’s output. Model call count and wall-clock latency are different quantities.

## P1 — Summarize independent notes

**Pattern:** Parallelization · **Time:** 10–15 minutes

Summarize three independent notes concurrently with `ThreadPoolExecutor`, using three workers. Collect results keyed by note ID, regardless of completion order. Return `{summaries, elapsed_seconds}`. Measure the call group, not notebook setup.

**Supplied model operations**

- `summary_A`, `summary_B`, `summary_C`: payload `{note}` → `{summary}`.
- Simulated delays differ intentionally. `model.calls` records start and finish times.

**Manual success criteria**

- Three calls overlap; inspect timestamps rather than relying on a strict timing assertion.
- All IDs map to the correct summaries, even though B finishes before A.
- Do not call `.result()` immediately after submitting each individual task; that would serialize the work.

In [ ]:
NOTES = [{'id':'A','text':'The support backlog shrank.'}, {'id':'B','text':'The new release shipped.'}, {'id':'C','text':'Hiring is paused.'}]
model = RecordedModel({
 'summary_A':[{'summary':'Support backlog decreased.'}],
 'summary_B':[{'summary':'Release delivered.'}],
 'summary_C':[{'summary':'Hiring paused.'}]
}, required={f'summary_{x}':['note'] for x in 'ABC'}, delays={'summary_A':0.30,'summary_B':0.10,'summary_C':0.20})

In [ ]:
def parallel_summaries(notes, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement P1")

# Run after implementing:
# model.reset()
# result = parallel_summaries(NOTES, model)
# print(result)
# print(model.calls)

**Explain after finishing:** Why does concurrency reduce latency here without reducing the number of model calls?

## P2 — Independent perspectives, then synthesis

**Pattern:** Parallelization · **Time:** 10–15 minutes

Ask an operations reviewer and a finance reviewer to examine the same proposal independently and concurrently. After both finish, make one synthesis call using both reviews. Return `{reviews, decision}`. Neither reviewer should see the other reviewer’s output.

**Supplied model operations**

- `ops_review`, `finance_review`: payload `{proposal}` → `{findings}`.
- `synthesize`: payload `{proposal, reviews}` → `{decision}`.

**Manual success criteria**

- Review calls overlap, and synthesis starts after both finish.
- Preserve reviewer labels in the synthesis payload.
- Do not present agreement between reviewers as proof the decision is correct.

In [ ]:
PROPOSAL = "Add weekend coverage for a three-month pilot."
model = RecordedModel({
 'ops_review':[{'findings':['Coverage may reduce weekend backlog.','Staff availability is unknown.']}],
 'finance_review':[{'findings':['Pilot cost needs an explicit budget.']}],
 'synthesize':[{'decision':'Define staffing availability and a capped budget before starting the pilot.'}]
}, required={'ops_review':['proposal'],'finance_review':['proposal'],'synthesize':['proposal','reviews']}, delays={'ops_review':0.20,'finance_review':0.20})

In [ ]:
def parallel_reviews(proposal, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement P2")

# Run after implementing:
# model.reset()
# result = parallel_reviews(PROPOSAL, model)
# print(result)
# print(model.calls)

**Explain after finishing:** Why must synthesis wait, while the two review calls can run concurrently?

## P3 — Preserve partial success

**Pattern:** Parallelization · **Time:** 10–15 minutes

Process three independent documents concurrently. One model call fails. Return `{results, errors}` keyed by document ID, retaining successful results rather than discarding the entire batch. Do not retry or synthesize in this drill; isolate concurrent collection and error handling.

**Supplied model operations**

- `extract_A`, `extract_B`, `extract_C`: payload `{document}` → `{fields}` or an exception.
- The B fixture raises `RuntimeError` intentionally.

**Manual success criteria**

- Keep A and C in results; put B in errors.
- Associate every future with its document ID before collecting it.
- A failed call must not become an empty successful extraction.

In [ ]:
DOCUMENTS = [{'id':'A','text':'Owner: Morgan'}, {'id':'B','text':'Unreadable document'}, {'id':'C','text':'Owner: Taylor'}]
model = RecordedModel({
 'extract_A':[{'fields':{'owner':'Morgan'}}],
 'extract_B':[RuntimeError('Simulated model timeout')],
 'extract_C':[{'fields':{'owner':'Taylor'}}]
}, required={f'extract_{x}':['document'] for x in 'ABC'}, delays={'extract_A':0.20,'extract_B':0.10,'extract_C':0.15})

In [ ]:
def parallel_extract(documents, model):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement P3")

# Run after implementing:
# model.reset()
# result = parallel_extract(DOCUMENTS, model)
# print(result)
# print(model.calls)

**Explain after finishing:** When would partial results be useful, and when would a downstream task require all documents?

# 6. Tool use

The model proposes an operation; Python validates and runs an allowed local function, then supplies the result to the model. A model mentioning a tool does not execute it. All tools below are local simulations; no messages, purchases, or external changes occur.

## T1 — Execute one local lookup

**Pattern:** Tool use · **Time:** 10–15 minutes

Answer a stock question using the supplied `get_stock` tool. First ask the model what tool to call, validate the name and exact argument keys, run the Python function, then send its result to the model for a final answer. Return `{tool_call, tool_result, answer}`. Never use `eval` or dispatch arbitrary names from globals.

**Supplied model operations**

- `decide`: payload `{question, tools}` → `{tool, args}`.
- `respond`: payload `{question, tool_call, tool_result}` → `{answer}`.
- Allowed tool: `get_stock(sku)`; it returns local inventory data.

**Manual success criteria**

- Execute the actual supplied function and pass its return value to `respond`.
- Reject unknown tools, missing/extra argument keys, and non-string SKUs.
- Do not hardcode the final inventory answer or let the model invent stock data.

In [ ]:
QUESTION = "How many KIT-7 units are available?"
INVENTORY = {'KIT-7':{'available':8},'KIT-9':{'available':0}}
def get_stock(sku):
    if sku not in INVENTORY:
        raise ValueError('Unknown SKU')
    return INVENTORY[sku]
TOOLS = {'get_stock': get_stock}
TOOL_DESCRIPTIONS = {'get_stock':{'args':['sku'],'description':'Look up available units for a SKU.'}}
model = RecordedModel({
 'decide':[{'tool':'get_stock','args':{'sku':'KIT-7'}}],
 'respond':[{'answer':'There are 8 KIT-7 units available.'}]
}, required={'decide':['question','tools'],'respond':['question','tool_call','tool_result']})

In [ ]:
def answer_stock_question(question, model, tools, tool_descriptions):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement T1")

# Run after implementing:
# model.reset()
# result = answer_stock_question(QUESTION, model, TOOLS, TOOL_DESCRIPTIONS)
# print(result)
# print(model.calls)

**Explain after finishing:** How does supplying a tool result differ from asking the model to guess the answer?

## T2 — Continue after a tool result

**Pattern:** Tool use · **Time:** 10–15 minutes

Run a bounded tool-use loop. The model may request a local account lookup, then a quote calculation using the returned discount, then provide a final answer. Append every action and observation to history. Allow at most four model calls, and raise `RuntimeError` if no final answer appears. Return `{answer, history}`.

**Supplied model operations**

- `next`: payload `{question, tools, history}` → either `{type:"tool", tool, args}` or `{type:"final", answer}`.
- Allowed tools: `lookup_account(account_id)` and `calculate_quote(list_price, discount)`.
- This fixture needs three model calls and two local tool calls.

**Manual success criteria**

- The second model call sees the account lookup observation; the final call sees the quote result.
- Validate tool names, exact argument keys and basic argument types before dispatch.
- Do not treat numeric Booleans as prices; require a nonnegative list price and discount between zero and one.
- Stop when the model returns a final answer or the model-call budget is exhausted.

In [ ]:
QUESTION = "What is the quote for account C-1 on an item listed at 100?"
def lookup_account(account_id):
    if account_id != 'C-1':
        raise ValueError('Unknown account')
    return {'discount':0.10}
def calculate_quote(list_price, discount):
    return {'total':round(list_price * (1 - discount), 2)}
TOOLS = {'lookup_account':lookup_account, 'calculate_quote':calculate_quote}
TOOL_DESCRIPTIONS = {'lookup_account':{'args':['account_id']}, 'calculate_quote':{'args':['list_price','discount']}}
model = RecordedModel({
 'next':[{'type':'tool','tool':'lookup_account','args':{'account_id':'C-1'}},
         {'type':'tool','tool':'calculate_quote','args':{'list_price':100,'discount':0.10}},
         {'type':'final','answer':'The quote is 90.00 after the account discount.'}]
}, required={'next':['question','tools','history']})

In [ ]:
def run_quote_loop(question, model, tools, tool_descriptions, max_calls=4):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement T2")

# Run after implementing:
# model.reset()
# result = run_quote_loop(QUESTION, model, TOOLS, TOOL_DESCRIPTIONS)
# print(result)
# print(model.calls)

**Explain after finishing:** Why can the two tool operations not safely run in parallel in this workflow?

## T3 — Reject a bad tool request and recover

**Pattern:** Tool use · **Time:** 10–15 minutes

The first model request names an unavailable tool. Record a structured error observation without executing it, then allow the model to recover by requesting an allowed lookup. Feed the successful result back before accepting a final answer. Allow at most four model calls. Return `{answer, history}` or raise explicitly if the budget expires.

**Supplied model operations**

- `next`: same action format as T2 and payload `{question, tools, history}`.
- Only `lookup_policy(topic)` is allowed. An unknown tool must produce an error observation listing available tools.
- Do not simulate successful execution of the invalid request.

**Manual success criteria**

- Make three model calls and execute only the single allowed lookup.
- History contains the invalid request, its error observation, the valid lookup and its real result.
- Reject malformed argument keys and record tool exceptions as errors, not factual observations.
- `TOOL_EXECUTIONS` should contain only `returns` after one run; clear it before repeating.

In [ ]:
QUESTION = "What is the return window for unopened accessories?"
TOOL_EXECUTIONS = []
def lookup_policy(topic):
    if topic != 'returns':
        raise ValueError('Unknown topic')
    TOOL_EXECUTIONS.append(topic)
    return {'return_window_days':30,'condition':'unopened accessories'}
TOOLS = {'lookup_policy':lookup_policy}
TOOL_DESCRIPTIONS = {'lookup_policy':{'args':['topic']}}
model = RecordedModel({
 'next':[{'type':'tool','tool':'delete_customer','args':{'customer_id':'C-9'}},
         {'type':'tool','tool':'lookup_policy','args':{'topic':'returns'}},
         {'type':'final','answer':'Unopened accessories have a 30-day return window.'}]
}, required={'next':['question','tools','history']})

In [ ]:
def run_policy_loop(question, model, tools, tool_descriptions, max_calls=4):
    # TODO: implement this pattern using the supplied inputs and model.
    raise NotImplementedError("Implement T3")

# Run after implementing:
# model.reset()
# result = run_policy_loop(QUESTION, model, TOOLS, TOOL_DESCRIPTIONS)
# print(result)
# print(model.calls)

**Explain after finishing:** What would go wrong if you accepted the model’s tool name without a whitelist?

# Personal debrief

After each exercise, write three short notes below its cells:

- **What I passed between calls:** name the actual fields and values.
- **Where I got stuck:** architecture, Python, parsing, concurrency, or validation.
- **Why this pattern helped:** explain the concrete purpose of the extra call or branch.

For your first session, complete **D1, R1, and F1** without a timer. Then try the remaining first exercises with a 10–15-minute timer. Move to the second and third exercise in each pattern after the first is comfortable. These drills intentionally reveal the target pattern; full assessment practice should eventually remove that hint.